In [ ]:
import sagemaker
from sagemaker.inputs import FileSystemInput
from sagemaker.tensorflow import TensorFlow
from sagemaker.debugger import TensorBoardOutputConfig

In [ ]:
# Docker Image Details
ACCOUNT_ID = ""
REGION = "us-east-1"
DOCKER_REPOSITORY = "streamswitch-docker-images"

# FSx details
FILE_SYSTEM_ID = "fs-0e822f3a2183d9f80"
MOUNT_POINT = "/a3yodamv" 
ACCESS_MODE = "ro"
FILE_SYSTEM_TYPE = "FSxLustre"

In [ ]:
custom_image_uri = f"{ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/{DOCKER_REPOSITORY}:gpu-optimized"
session = sagemaker.Session()

# Define the FileSystemInput
fsx_input = FileSystemInput(
    file_system_id=FILE_SYSTEM_ID,
    file_system_type=FILE_SYSTEM_TYPE,
    directory_path=MOUNT_POINT,
    file_system_access_mode=ACCESS_MODE
)

tensorboard_output_config = TensorBoardOutputConfig(
    s3_output_path='s3://streamswitch-training-results',
    container_local_output_path='/opt/ml/output/tensorboard'
)

In [ ]:
estimator = TensorFlow(
    sagemaker_session=session,
    entry_point='streamswitch.py',
    source_dir='.',
    base_job_name="streamswitch-training-job",
    image_uri=custom_image_uri,
    role=sagemaker.get_execution_role(),
    instance_count=1,
    instance_type="ml.g5.4xlarge",
    # instance_type='ml.p3.2xlarge',|
    subnets=['subnet-099c27aa0c4534730'],
    security_group_ids=['sg-01a1363df4a019c30'],
    tensorboard_output_config=tensorboard_output_config,
    run_tensorboard_locally=True,
    enable_remote_debug=True,
    environment={'PYTHONUNBUFFERED': '1'}
    # output_path='s3://streamswitch-training-data',
)

estimator.fit({'train': fsx_input})